# Multimodal AI Pipeline for Legacy Geological Documents and Petrographic Image Analysis

**Portfolio project — executable Google Colab notebook**

This notebook demonstrates two connected workflows:

1. **Legacy Geological Document Understanding** — mixed files → searchable/scanned page handling → OCR/layout/table/entity extraction → page/document classification → traceable structured records.
2. **Petrographic Image Understanding** — image QC → semantic segmentation → instance separation → object features → object classification → uncertainty → quantitative petrography.

The final data model preserves the chain:

`source_document → page/sheet → detected_region → geological_image → object_id → prediction → confidence → measurements`

> **Scientific caution.** Synthetic data are used by default to make the notebook reproducible. Performance on synthetic data is *not* evidence of field performance. Real deployment requires source-level splitting, expert labels, domain-shift testing, and geological validation.

## 0. Colab installation
The first cell installs only lightweight dependencies required by the default path. Tesseract is optional: if unavailable, synthetic ground-truth text is used for evaluation and the OCR stage reports the fallback explicitly.

In [ ]:
# Colab-friendly dependencies. Re-running is safe.
import sys, subprocess, importlib.util

def pip_install(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)

required = {
    'fitz':'PyMuPDF', 'pdfplumber':'pdfplumber', 'docx':'python-docx',
    'openpyxl':'openpyxl', 'cv2':'opencv-python-headless', 'skimage':'scikit-image',
    'sklearn':'scikit-learn', 'scipy':'scipy', 'PIL':'Pillow', 'albumentations':'albumentations',
    'timm':'timm', 'reportlab':'reportlab'
}
missing=[pkg for mod,pkg in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing:', missing)
    pip_install(missing)
else:
    print('Core dependencies already available.')

# Optional system OCR. This may be unavailable outside Colab; notebook has a fallback.
try:
    subprocess.run(['tesseract','--version'], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print('Tesseract available.')
except Exception:
    print('Tesseract not found. In Colab you may run: !apt-get -qq update && apt-get -qq install -y tesseract-ocr')

## 1. Environment and central configuration
**Objective:** one configuration controls synthetic/folder mode, runtime, image size, epochs, confidence threshold and device. Reproducible seeds are applied to Python, NumPy and PyTorch.

In [ ]:
from pathlib import Path
import os, json, re, random, logging, hashlib, math, time, warnings, shutil
from dataclasses import dataclass, asdict
from typing import Optional, List, Dict, Tuple, Any
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont, ImageEnhance, ImageFilter, ImageOps
import cv2
from scipy import ndimage as ndi
from skimage import measure, morphology, segmentation, feature, exposure
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

@dataclass
class Config:
    DATA_SOURCE: str = 'synthetic'   # 'synthetic' or 'folder'
    DATA_DIR: str = '/content/geological_data'
    OUTPUT_DIR: str = '/content/geological_ai_results'
    FAST_DEMO: bool = True
    RANDOM_SEED: int = 42
    IMAGE_SIZE: int = 160
    NUM_SYNTHETIC_DOCUMENTS: int = 12
    NUM_SYNTHETIC_IMAGES: int = 48
    NUM_EPOCHS: int = 3
    CONFIDENCE_THRESHOLD: float = 0.65
    BATCH_SIZE: int = 8
    NUM_WORKERS: int = 0

CFG=Config()
if not Path('/content').exists():
    CFG.DATA_DIR='/mnt/data/geological_data'
    CFG.OUTPUT_DIR='/mnt/data/geological_ai_results'
if not CFG.FAST_DEMO:
    CFG.IMAGE_SIZE=224; CFG.NUM_SYNTHETIC_DOCUMENTS=30; CFG.NUM_SYNTHETIC_IMAGES=160; CFG.NUM_EPOCHS=12
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
Path(CFG.DATA_DIR).mkdir(parents=True, exist_ok=True); Path(CFG.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
seed_everything(CFG.RANDOM_SEED)
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(message)s')
print(asdict(CFG)); print('DEVICE =', DEVICE)

### Optional Google Drive mount
Uncomment only when needed. The notebook does **not** require Drive.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# CFG.DATA_DIR = '/content/drive/MyDrive/geological_data'

## 2. Synthetic multimodal dataset
**Geological meaning:** we need controlled ground truth for OCR, entities, semantic masks, instance masks and object classes. The generator creates quartz/carbonate grains, clay/cement matrix, blue epoxy-like pores/vugs and thin fractures. It also writes searchable PDFs, rasterized scanned PDFs, DOCX, XLSX and standalone images into the same folder structure expected in folder mode.

**Class IDs:** 0 matrix/background, 1 grain, 2 pore/vug, 3 fracture, 4 cement/clay. Instance IDs are unique per object.

In [ ]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from docx import Document
from openpyxl import Workbook

SEMANTIC_NAMES={0:'matrix',1:'grain',2:'pore',3:'fracture',4:'cement_clay'}
OBJECT_NAMES={1:'quartz_grain',2:'carbonate_grain',3:'clay_cement',4:'interparticle_pore',5:'isolated_vug',6:'connected_vug',7:'microfracture'}
PALETTE=np.array([[85,75,65],[205,190,155],[45,125,205],[220,70,70],[135,110,80]],dtype=np.uint8)

def irregular_polygon(cx,cy,r,n=12,rng=None):
    rng=rng or np.random.default_rng(); ang=np.sort(rng.uniform(0,2*np.pi,n)); rr=r*rng.uniform(.65,1.25,n)
    return np.c_[cx+rr*np.cos(ang),cy+rr*np.sin(ang)].astype(np.int32)

def synth_petrographic(idx, size=224, seed=42):
    rng=np.random.default_rng(seed+idx); img=np.zeros((size,size,3),np.uint8); img[:]=[120,105,85]
    sem=np.zeros((size,size),np.uint8); inst=np.zeros((size,size),np.int32); objects=[]; oid=1
    # textured matrix
    noise=rng.normal(0,10,(size,size,1)); img=np.clip(img+noise,0,255).astype(np.uint8)
    # grains
    for _ in range(rng.integers(14,25)):
        cx,cy=rng.integers(18,size-18,2); r=int(rng.integers(9,25)); cls=int(rng.choice([1,2])); poly=irregular_polygon(cx,cy,r,int(rng.integers(8,15)),rng)
        mask=np.zeros((size,size),np.uint8); cv2.fillPoly(mask,[poly],1)
        color=(205,190,155) if cls==1 else (185,170,135)
        tex=rng.normal(0,8,(size,size,1)); layer=np.clip(np.array(color)+tex,0,255).astype(np.uint8)
        img[mask>0]=layer[mask>0]; sem[mask>0]=1; inst[mask>0]=oid
        ys,xs=np.where(mask>0); objects.append({'object_id':oid,'class_id':cls,'class_name':OBJECT_NAMES[cls],'bbox':[int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())]}); oid+=1
    # cement/clay patches
    for _ in range(rng.integers(2,5)):
        cx,cy=rng.integers(20,size-20,2); axes=tuple(map(int,rng.integers(8,22,2))); mask=np.zeros((size,size),np.uint8)
        cv2.ellipse(mask,(int(cx),int(cy)),axes,float(rng.uniform(0,180)),0,360,1,-1); img[mask>0]=[135,110,80]; sem[mask>0]=4; inst[mask>0]=oid
        ys,xs=np.where(mask>0); objects.append({'object_id':oid,'class_id':3,'class_name':OBJECT_NAMES[3],'bbox':[int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())]}); oid+=1
    # pores/vugs, sometimes touching
    for j in range(rng.integers(7,14)):
        cls=int(rng.choice([4,5,6],p=[.45,.35,.20])); cx,cy=rng.integers(15,size-15,2); axes=tuple(map(int,rng.integers(5,18,2))); mask=np.zeros((size,size),np.uint8)
        cv2.ellipse(mask,(int(cx),int(cy)),axes,float(rng.uniform(0,180)),0,360,1,-1)
        if cls==6:
            cv2.ellipse(mask,(min(size-1,int(cx+axes[0]*.8)),int(cy)),axes,float(rng.uniform(0,180)),0,360,1,-1)
        img[mask>0]=[40,120,210]; sem[mask>0]=2; inst[mask>0]=oid
        ys,xs=np.where(mask>0); objects.append({'object_id':oid,'class_id':cls,'class_name':OBJECT_NAMES[cls],'bbox':[int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())]}); oid+=1
    # fractures
    for _ in range(rng.integers(1,4)):
        x1,y1=rng.integers(5,size-5,2); angle=rng.uniform(0,np.pi); L=int(rng.integers(size//4,size//2)); x2=int(np.clip(x1+L*np.cos(angle),2,size-3)); y2=int(np.clip(y1+L*np.sin(angle),2,size-3)); w=int(rng.integers(1,4)); mask=np.zeros((size,size),np.uint8)
        cv2.line(mask,(int(x1),int(y1)),(x2,y2),1,w); img[mask>0]=[225,65,65]; sem[mask>0]=3; inst[mask>0]=oid
        ys,xs=np.where(mask>0); objects.append({'object_id':oid,'class_id':7,'class_name':OBJECT_NAMES[7],'bbox':[int(xs.min()),int(ys.min()),int(xs.max()),int(ys.max())]}); oid+=1
    # illumination/noise
    yy,xx=np.mgrid[:size,:size]; illum=(1+0.12*((xx-size/2)/size)+0.08*((yy-size/2)/size))[...,None]
    img=np.clip(img*illum+rng.normal(0,4,img.shape),0,255).astype(np.uint8)
    porosity=float((sem==2).mean())
    return img,sem,inst,objects,{'image_id':f'petro_{idx:04d}','gt_porosity_2d':porosity,'n_objects':len(objects)}

def degrade_scan(pil, seed):
    rng=np.random.default_rng(seed); angle=float(rng.uniform(-3,3)); out=pil.rotate(angle,expand=False,fillcolor='white')
    if rng.random()<.8: out=out.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(.3,1.2))))
    arr=np.array(out).astype(np.float32); h,w=arr.shape[:2]; grad=np.linspace(rng.uniform(.8,1),rng.uniform(.95,1.15),w)[None,:,None]
    arr=np.clip(arr*grad+rng.normal(0,rng.uniform(2,10),arr.shape),0,255).astype(np.uint8)
    return Image.fromarray(arr)

def make_page_image(text, table_rows, size=(1275,1650), seed=0):
    im=Image.new('RGB',size,'white'); d=ImageDraw.Draw(im); font=ImageFont.load_default(); y=70
    d.text((70,y),'LEGACY GEOLOGICAL REPORT',fill='black',font=font); y+=45
    for line in text.split('\n'):
        d.text((70,y),line,fill='black',font=font); y+=28
    y+=25; x0=70; widths=[170,150,160,150,130,140,140,160]; headers=['Well','Depth_m','Formation','Lithology','Porosity_%','Perm_mD','Sw_%','Sample']
    xs=[x0]
    for w in widths: xs.append(xs[-1]+w)
    rowh=34
    for r in range(len(table_rows)+2): d.line((x0,y+r*rowh,xs[-1],y+r*rowh),fill='black',width=1)
    for x in xs: d.line((x,y,x,y+(len(table_rows)+1)*rowh),fill='black',width=1)
    for j,h in enumerate(headers): d.text((xs[j]+4,y+8),h,fill='black',font=font)
    for i,row in enumerate(table_rows):
        for j,val in enumerate(row): d.text((xs[j]+4,y+(i+1)*rowh+8),str(val),fill='black',font=font)
    return degrade_scan(im,seed)

def generate_synthetic_dataset(cfg=CFG):
    root=Path(cfg.DATA_DIR); shutil.rmtree(root,ignore_errors=True)
    for sub in ['documents','images','annotations','metadata']: (root/sub).mkdir(parents=True,exist_ok=True)
    image_records=[]; object_records=[]
    for i in range(cfg.NUM_SYNTHETIC_IMAGES):
        img,sem,inst,objs,meta=synth_petrographic(i,cfg.IMAGE_SIZE,cfg.RANDOM_SEED)
        Image.fromarray(img).save(root/'images'/f'petro_{i:04d}.png'); np.save(root/'annotations'/f'petro_{i:04d}_semantic.npy',sem); np.save(root/'annotations'/f'petro_{i:04d}_instance.npy',inst)
        meta.update({'path':str(root/'images'/f'petro_{i:04d}.png'),'source_group':f'source_{i//3:03d}'})
        image_records.append(meta)
        for o in objs: object_records.append({**o,'image_id':meta['image_id'],'source_group':meta['source_group']})
    pd.DataFrame(image_records).to_csv(root/'metadata'/'image_metadata.csv',index=False); pd.DataFrame(object_records).to_csv(root/'metadata'/'object_ground_truth.csv',index=False)
    formations=['Arab-D','Buwaib','Hanifa','Khuff','Wasia']; lith=['limestone','dolomite','sandstone','shale']
    doc_gt=[]
    for i in range(cfg.NUM_SYNTHETIC_DOCUMENTS):
        well=f'WELL-{i+1:03d}'; formation=formations[i%len(formations)]; litho=lith[i%len(lith)]; depth=1800+i*37
        txt=f'Well: {well}\nField: SYNTHETIC FIELD\nFormation: {formation}\nDepth interval: {depth}-{depth+25} m\nLithology: {litho}\nPorosity: {12+i%9}%\nPermeability: {5+i*3} mD\nWater saturation: {25+i%20}%\nSample ID: S-{i:03d}\nDescription: Petrographic examination notes pores, grains and microfractures.'
        rows=[[well,depth,formation,litho,12+i%9,5+i*3,25+i%20,f'S-{i:03d}']]
        # searchable PDF
        pdf=root/'documents'/f'report_{i:03d}_searchable.pdf'; c=canvas.Canvas(str(pdf),pagesize=letter); yy=740
        for line in txt.split('\n'): c.drawString(60,yy,line); yy-=22
        c.save(); doc_gt.append({'file':str(pdf),'text':txt,'doc_class':'geological_report','well':well,'formation':formation})
        # scanned PDF every third doc
        if i%3==0:
            page=make_page_image(txt,rows,seed=cfg.RANDOM_SEED+i); scanpdf=root/'documents'/f'report_{i:03d}_scan.pdf'; page.save(scanpdf,'PDF',resolution=150.0); doc_gt.append({'file':str(scanpdf),'text':txt,'doc_class':'geological_report','well':well,'formation':formation})
        if i<3:
            d=Document(); d.add_heading('Core Description',0); d.add_paragraph(txt); t=d.add_table(rows=2,cols=4); vals=['Well',well,'Formation',formation,'Depth',str(depth),'Lithology',litho]
            for k,v in enumerate(vals): t.cell(k//4,k%4).text=str(v)
            d.save(root/'documents'/f'core_{i:03d}.docx')
    wb=Workbook(); ws=wb.active; ws.title='LabData'; ws.append(['Well','Depth_m','Formation','Porosity_pct','Perm_mD','Sw_pct'])
    for i in range(cfg.NUM_SYNTHETIC_DOCUMENTS): ws.append([f'WELL-{i+1:03d}',1800+i*37,formations[i%5],12+i%9,5+i*3,25+i%20])
    wb.save(root/'documents'/'laboratory.xlsx'); pd.DataFrame(doc_gt).to_json(root/'metadata'/'document_ground_truth.json',orient='records',indent=2)
    print('Synthetic dataset:',root); return root

if CFG.DATA_SOURCE=='synthetic':
    generate_synthetic_dataset(CFG)

### Visual check of synthetic petrography ground truth

In [ ]:
meta=pd.read_csv(Path(CFG.DATA_DIR)/'metadata'/'image_metadata.csv'); p=Path(meta.iloc[0].path); img=np.array(Image.open(p)); sem=np.load(Path(CFG.DATA_DIR)/'annotations'/f'{p.stem}_semantic.npy'); inst=np.load(Path(CFG.DATA_DIR)/'annotations'/f'{p.stem}_instance.npy')
fig,ax=plt.subplots(1,3,figsize=(13,4)); ax[0].imshow(img); ax[0].set_title('Synthetic RGB'); ax[1].imshow(PALETTE[sem]); ax[1].set_title('Semantic mask'); ax[2].imshow(inst,cmap='nipy_spectral'); ax[2].set_title('Instance mask')
for a in ax:a.axis('off'); plt.tight_layout(); plt.savefig(Path(CFG.OUTPUT_DIR)/'synthetic_example.png',dpi=160); plt.show()
print('GT visible porosity:',round(float((sem==2).mean()),3),'| instances:',int(inst.max()))

## 3. File discovery, validation and unified manifest
Corrupted/unsupported files are logged rather than crashing the pipeline. SHA-256 hashes support traceability and duplicate detection.

In [ ]:
SUPPORTED={'.pdf','.docx','.xlsx','.xls','.csv','.jpg','.jpeg','.png','.tif','.tiff'}
def sha256_file(path,chunk=1<<20):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while b:=f.read(chunk): h.update(b)
    return h.hexdigest()

def discover_files(root):
    rows=[]
    for p in sorted(Path(root).rglob('*')):
        if not p.is_file() or 'annotations' in p.parts or 'metadata' in p.parts: continue
        rec={'file_path':str(p),'extension':p.suffix.lower(),'size_bytes':p.stat().st_size,'hash':'','status':'pending','pages_or_sheets':None,'content_type':'unsupported','error':''}
        if p.suffix.lower() not in SUPPORTED: rec['status']='unsupported'; rows.append(rec); continue
        try:
            rec['hash']=sha256_file(p); rec['content_type']='document' if p.suffix.lower() in {'.pdf','.docx','.xlsx','.xls','.csv'} else 'image'; rec['status']='valid'
        except Exception as e: rec['status']='error'; rec['error']=str(e)
        rows.append(rec)
    df=pd.DataFrame(rows); df.to_csv(Path(CFG.OUTPUT_DIR)/'document_manifest.csv',index=False); return df
manifest=discover_files(CFG.DATA_DIR); display(manifest.head(12)); print(manifest.extension.value_counts())

## 4. Document ingestion
PDF pages are classified as **searchable** when an active text layer contains enough characters; otherwise the page is rasterized for OCR. Word paragraphs/tables and Excel sheets are preserved. Standalone images become image records.

In [ ]:
import fitz, pdfplumber
from docx import Document as DocxDocument

def pdf_pages(path,dpi=150):
    out=[]; doc=fitz.open(path)
    for i,page in enumerate(doc):
        text=page.get_text('text').strip(); searchable=len(re.sub(r'\s+','',text))>=25
        pix=page.get_pixmap(matrix=fitz.Matrix(dpi/72,dpi/72),alpha=False); arr=np.frombuffer(pix.samples,np.uint8).reshape(pix.height,pix.width,pix.n)
        if pix.n==4: arr=cv2.cvtColor(arr,cv2.COLOR_RGBA2RGB)
        out.append({'source_file':str(path),'page':i+1,'text_layer':text,'searchable':searchable,'image':arr,'kind':'pdf_page'})
    return out

def read_docx(path):
    d=DocxDocument(path); text='\n'.join(p.text for p in d.paragraphs if p.text.strip()); tables=[]
    for t in d.tables: tables.append([[c.text for c in row.cells] for row in t.rows])
    return [{'source_file':str(path),'page':1,'text_layer':text,'searchable':True,'image':None,'kind':'docx','tables':tables}]

def read_excel(path):
    book=pd.ExcelFile(path); out=[]
    for j,s in enumerate(book.sheet_names):
        df=pd.read_excel(path,sheet_name=s); out.append({'source_file':str(path),'page':j+1,'sheet':s,'text_layer':df.to_csv(index=False),'searchable':True,'image':None,'kind':'excel','tables':[df.values.tolist()]})
    return out

def read_csv(path):
    df=pd.read_csv(path); return [{'source_file':str(path),'page':1,'sheet':'csv','text_layer':df.to_csv(index=False),'searchable':True,'image':None,'kind':'csv','tables':[df.values.tolist()]}]

def read_image(path):
    im=ImageOps.exif_transpose(Image.open(path)).convert('RGB'); return [{'source_file':str(path),'page':1,'text_layer':'','searchable':False,'image':np.array(im),'kind':'standalone_image'}]

def ingest_manifest(df):
    records=[]
    for _,r in df[df.status=='valid'].iterrows():
        p=Path(r.file_path)
        try:
            if p.suffix.lower()=='.pdf': records+=pdf_pages(p)
            elif p.suffix.lower()=='.docx': records+=read_docx(p)
            elif p.suffix.lower() in {'.xlsx','.xls'}: records+=read_excel(p)
            elif p.suffix.lower()=='.csv': records+=read_csv(p)
            else: records+=read_image(p)
        except Exception as e: logging.warning('Failed %s: %s',p,e)
    return records
records=ingest_manifest(manifest); print('Unified records:',len(records)); print(pd.DataFrame([{k:v for k,v in r.items() if k not in ['image','tables']} for r in records]).head())

## 5. Scan preprocessing and objective quality control
We compare raw grayscale, CLAHE, denoised/adaptive-threshold variants. OCR confidence is preferred when Tesseract is available; otherwise a composite contrast/sharpness score selects the variant. This avoids blindly applying one preprocessing recipe.

In [ ]:
def estimate_skew(gray):
    bw=cv2.threshold(gray,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]; pts=np.column_stack(np.where(bw>0))
    if len(pts)<20:return 0.0
    angle=cv2.minAreaRect(pts[:,::-1].astype(np.float32))[-1]; return float(-(90+angle) if angle<-45 else -angle)

def rotate_bound(img,angle):
    h,w=img.shape[:2]; M=cv2.getRotationMatrix2D((w/2,h/2),angle,1); return cv2.warpAffine(img,M,(w,h),flags=cv2.INTER_CUBIC,borderMode=cv2.BORDER_CONSTANT,borderValue=255)

def quality_metrics(img):
    g=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY) if img.ndim==3 else img
    return {'blur_score':float(cv2.Laplacian(g,cv2.CV_64F).var()),'brightness':float(g.mean()),'contrast':float(g.std()),'skew_deg':estimate_skew(g)}

def preprocess_variants(rgb):
    g=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY); skew=estimate_skew(g); desk=rotate_bound(g,skew)
    clahe=cv2.createCLAHE(2.0,(8,8)).apply(desk); den=cv2.fastNlMeansDenoising(clahe,None,10,7,21); th=cv2.adaptiveThreshold(den,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,31,15)
    return {'gray':g,'clahe':clahe,'denoised':den,'binary':th}

def choose_variant(rgb):
    vs=preprocess_variants(rgb); scores={}
    for k,v in vs.items(): scores[k]=float(v.std()+0.02*cv2.Laplacian(v,cv2.CV_64F).var())
    best=max(scores,key=scores.get); return best,vs[best],vs,scores
scan=next((r for r in records if r['kind']=='pdf_page' and not r['searchable']),None)
if scan:
    best,corr,vs,scores=choose_variant(scan['image']); print('QC:',quality_metrics(scan['image']),'selected:',best,scores)
    fig,ax=plt.subplots(1,4,figsize=(15,5));
    for a,(k,v) in zip(ax,vs.items()): a.imshow(v,cmap='gray'); a.set_title(k); a.axis('off')
    plt.tight_layout(); plt.savefig(Path(CFG.OUTPUT_DIR)/'scan_preprocessing.png',dpi=150); plt.show()

## 6. Lightweight page-layout analysis
A production system can substitute LayoutLMv3/DocLayNet/Docling. The default fallback is deterministic OpenCV morphology: long horizontal/vertical line structures indicate tables; connected text-like components form paragraph regions. Layout is required because OCR alone loses page structure.

In [ ]:
def layout_fallback(rgb):
    g=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY); bw=cv2.threshold(g,0,255,cv2.THRESH_BINARY_INV+cv2.THRESH_OTSU)[1]
    hk=cv2.getStructuringElement(cv2.MORPH_RECT,(max(20,rgb.shape[1]//20),1)); vk=cv2.getStructuringElement(cv2.MORPH_RECT,(1,max(20,rgb.shape[0]//30)))
    lines=cv2.bitwise_or(cv2.morphologyEx(bw,cv2.MORPH_OPEN,hk),cv2.morphologyEx(bw,cv2.MORPH_OPEN,vk))
    regs=[]
    for cnt in cv2.findContours(lines,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
        x,y,w,h=cv2.boundingRect(cnt)
        if w*h>2000: regs.append({'label':'table_or_chart','bbox':[x,y,x+w,y+h],'confidence':0.65})
    # text blocks
    dil=cv2.dilate(bw,cv2.getStructuringElement(cv2.MORPH_RECT,(15,5)),iterations=1)
    for cnt in cv2.findContours(dil,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)[0]:
        x,y,w,h=cv2.boundingRect(cnt)
        if w>80 and 8<h<200 and w*h>1200: regs.append({'label':'paragraph','bbox':[x,y,x+w,y+h],'confidence':0.55})
    return regs

def draw_regions(rgb,regs):
    out=rgb.copy()
    for r in regs:
        x1,y1,x2,y2=r['bbox']; cv2.rectangle(out,(x1,y1),(x2,y2),(255,0,0),2); cv2.putText(out,r['label'],(x1,max(15,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,.5,(255,0,0),1)
    return out
if scan:
    regs=layout_fallback(scan['image']); vis=draw_regions(scan['image'],regs); plt.figure(figsize=(10,8)); plt.imshow(vis); plt.axis('off'); plt.title('Fallback layout regions'); plt.show(); print(regs[:8])

## 7. OCR and scanned-table reconstruction
Searchable PDFs bypass OCR. Scanned pages use Tesseract when installed. OCR tokens retain boxes/confidence. A morphology-based table routine detects grid cells and assigns OCR tokens by token-center coordinates. Synthetic mode can calculate CER/WER only when OCR is actually available.

In [ ]:
import importlib.util
HAS_TESS=shutil.which('tesseract') is not None and importlib.util.find_spec('pytesseract') is not None
if HAS_TESS: import pytesseract

def ocr_image(rgb):
    if not HAS_TESS: return {'text':'','tokens':pd.DataFrame(),'mean_confidence':0.0,'method':'unavailable'}
    _,best,_,_=choose_variant(rgb); df=pytesseract.image_to_data(best,output_type=pytesseract.Output.DATAFRAME)
    df=df.dropna(subset=['text']); df=df[df.text.astype(str).str.strip()!='']; conf=pd.to_numeric(df.conf,errors='coerce').fillna(-1); df=df[conf>=0].copy(); df['conf']=pd.to_numeric(df.conf,errors='coerce')
    return {'text':' '.join(df.text.astype(str)),'tokens':df,'mean_confidence':float(df.conf.mean()) if len(df) else 0.0,'method':'tesseract'}

def levenshtein(a,b):
    prev=list(range(len(b)+1))
    for i,ca in enumerate(a,1):
        cur=[i]
        for j,cb in enumerate(b,1): cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(ca!=cb)))
        prev=cur
    return prev[-1]
def cer(ref,hyp): return levenshtein(ref,hyp)/max(1,len(ref))
def wer(ref,hyp): return levenshtein(ref.split(),hyp.split())/max(1,len(ref.split()))

def extract_text_record(r):
    if r['searchable']: return {'text':r['text_layer'],'confidence':1.0,'method':'text_layer','tokens':pd.DataFrame()}
    if r['image'] is not None:
        o=ocr_image(r['image'])
        # explicit synthetic fallback, never represented as OCR accuracy
        if not o['text'] and CFG.DATA_SOURCE=='synthetic': o['text']=''; o['method']='ocr_unavailable'
        return {'text':o['text'],'confidence':o['mean_confidence']/100 if o['mean_confidence']>1 else o['mean_confidence'],'method':o['method'],'tokens':o['tokens']}
    return {'text':'','confidence':0,'method':'none','tokens':pd.DataFrame()}
text_rows=[]
for r in records:
    o=extract_text_record(r); r['extracted_text']=o['text']; r['ocr_confidence']=o['confidence']; r['ocr_method']=o['method']; r['ocr_tokens']=o['tokens']
    text_rows.append({'source_file':r['source_file'],'page':r['page'],'text':o['text'],'confidence':o['confidence'],'method':o['method']})
with open(Path(CFG.OUTPUT_DIR)/'extracted_text.json','w') as f: json.dump(text_rows,f,indent=2)
display(pd.DataFrame(text_rows).head())

## 8. Geological entity extraction and normalization
A transparent baseline uses regex + dictionaries + unit normalization. Every value retains provenance, original text, normalized value, confidence and extraction method.

In [ ]:
FORMATION_ALIASES={'arab d':'Arab-D','arab-d':'Arab-D','buwaib':'Buwaib','hanifa':'Hanifa','khuff':'Khuff','wasia':'Wasia'}
patterns={
'well':r'(?i)\bwell\s*[:#-]?\s*([A-Z0-9-]+)',
'formation':r'(?i)\bformation\s*[:#-]?\s*([A-Za-z0-9 -]+)',
'depth_interval':r'(?i)\bdepth(?: interval)?\s*[:#-]?\s*([0-9.,]+)\s*[-–]\s*([0-9.,]+)\s*(m|ft)',
'lithology':r'(?i)\blithology\s*[:#-]?\s*([A-Za-z -]+)',
'porosity':r'(?i)\bporosity\s*[:#-]?\s*([0-9.,]+)\s*%',
'permeability':r'(?i)\bpermeability\s*[:#-]?\s*([0-9.,]+)\s*(mD|D)',
'water_saturation':r'(?i)\bwater saturation\s*[:#-]?\s*([0-9.,]+)\s*%',
'sample_id':r'(?i)\bsample(?: id)?\s*[:#-]?\s*([A-Z0-9-]+)'}
def num(s): return float(str(s).replace(',','.'))
def extract_entities(text,source,page,base_conf=1.0):
    rows=[]
    for ent,pat in patterns.items():
        for m in re.finditer(pat,text):
            raw=m.group(0); val=m.groups()
            norm=val[0]
            if ent=='formation': norm=FORMATION_ALIASES.get(val[0].strip().lower(),val[0].strip())
            elif ent=='depth_interval':
                a,b,u=num(val[0]),num(val[1]),val[2].lower(); norm={'top_m':a*.3048 if u=='ft' else a,'base_m':b*.3048 if u=='ft' else b}
            elif ent in {'porosity','water_saturation'}: norm=num(val[0])/100
            elif ent=='permeability': norm=num(val[0])*(1000 if val[1].lower()=='d' else 1)
            else: norm=str(norm).strip()
            rows.append({'source_file':source,'page_or_sheet':page,'entity':ent,'original_text':raw,'normalized_value':json.dumps(norm) if isinstance(norm,dict) else norm,'confidence':float(base_conf*.95),'method':'regex_rules'})
    return rows
entity_rows=[]
for r in records: entity_rows+=extract_entities(r.get('extracted_text') or r.get('text_layer',''),r['source_file'],r['page'],r.get('ocr_confidence',1.0) or .5)
entities=pd.DataFrame(entity_rows); entities.to_csv(Path(CFG.OUTPUT_DIR)/'geological_entities.csv',index=False); display(entities.head(15))

## 9. Page/document classification — TF-IDF baseline
This is **document classification**, not OCR and not segmentation. We classify a page/document from its textual content. Metrics are reported only after a source-level split. In tiny synthetic demos, labels are generated from file type/content; real projects require human labels.

In [ ]:
def heuristic_doc_label(r):
    name=Path(r['source_file']).name.lower(); txt=(r.get('extracted_text') or r.get('text_layer','')).lower()
    if 'laboratory' in name or r['kind'] in {'excel','csv'}: return 'laboratory_analysis'
    if 'core_' in name: return 'core_description'
    if 'petro' in txt: return 'petrographic_report'
    if r['kind']=='standalone_image': return 'map_or_figure'
    return 'geological_report'
page_df=pd.DataFrame([{'source_file':r['source_file'],'page':r['page'],'text':r.get('extracted_text') or r.get('text_layer',''),'label':heuristic_doc_label(r)} for r in records if (r.get('extracted_text') or r.get('text_layer','')).strip()])
if page_df.label.nunique()>1 and len(page_df)>=8:
    groups=page_df.source_file; gss=GroupShuffleSplit(1,test_size=.3,random_state=CFG.RANDOM_SEED); tr,te=next(gss.split(page_df,page_df.label,groups))
    vec=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=3000); Xtr=vec.fit_transform(page_df.iloc[tr].text); Xte=vec.transform(page_df.iloc[te].text)
    clf=LogisticRegression(max_iter=1000,class_weight='balanced').fit(Xtr,page_df.iloc[tr].label); pred=clf.predict(Xte)
    print(classification_report(page_df.iloc[te].label,pred,zero_division=0)); page_df.loc[te,'predicted_class']=pred
else: page_df['predicted_class']=page_df.label
page_df.to_csv(Path(CFG.OUTPUT_DIR)/'page_classification.csv',index=False); display(page_df.head())

## 10. Geological-image detection
For the synthetic portfolio path, standalone petrographic images are known geological images. For PDF pages, the fallback proposes large non-text regions as figure candidates. Production deployment should replace this with a labeled figure-type detector.

In [ ]:
def geological_image_candidates(records):
    rows=[]
    for r in records:
        if r['kind']=='standalone_image': rows.append({'source_file':r['source_file'],'page':r['page'],'image_id':Path(r['source_file']).stem,'image_path':r['source_file'],'figure_type':'petrographic_or_core','confidence':1.0})
    return pd.DataFrame(rows)
geo_images=geological_image_candidates(records); display(geo_images.head())

## 11. Leakage-safe semantic segmentation split
**Critical rule:** split by source image/group **before** generating augmented patches. The synthetic generator assigns three related images to one `source_group`; GroupShuffleSplit ensures related sources never cross train/validation/test.

In [ ]:
image_meta=pd.read_csv(Path(CFG.DATA_DIR)/'metadata'/'image_metadata.csv')
g1=GroupShuffleSplit(1,test_size=.30,random_state=CFG.RANDOM_SEED); train_idx,tmp_idx=next(g1.split(image_meta,groups=image_meta.source_group)); tmp=image_meta.iloc[tmp_idx].reset_index(drop=True)
g2=GroupShuffleSplit(1,test_size=.50,random_state=CFG.RANDOM_SEED+1); val_rel,test_rel=next(g2.split(tmp,groups=tmp.source_group))
splits={'train':image_meta.iloc[train_idx].reset_index(drop=True),'val':tmp.iloc[val_rel].reset_index(drop=True),'test':tmp.iloc[test_rel].reset_index(drop=True)}
for k,v in splits.items(): print(k,len(v),'groups',v.source_group.nunique())
assert set(splits['train'].source_group).isdisjoint(splits['val'].source_group) and set(splits['train'].source_group).isdisjoint(splits['test'].source_group)

## 12. Lightweight U-Net semantic segmentation
Semantic segmentation answers: **“Which class does each pixel belong to?”** It does *not* distinguish Grain 1 from Grain 2. We train on five pixel classes with Cross-Entropy + soft Dice loss.

In [ ]:
class PetroDataset(Dataset):
    def __init__(self,df,size): self.df=df; self.size=size
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        p=Path(self.df.iloc[i].path); im=np.array(Image.open(p).convert('RGB').resize((self.size,self.size),Image.Resampling.BILINEAR)); m=np.load(Path(CFG.DATA_DIR)/'annotations'/f'{p.stem}_semantic.npy'); m=np.array(Image.fromarray(m).resize((self.size,self.size),Image.Resampling.NEAREST))
        x=torch.from_numpy(im.transpose(2,0,1)).float()/255.; y=torch.from_numpy(m).long(); return x,y,p.stem
class ConvBlock(nn.Module):
    def __init__(self,a,b): super().__init__(); self.net=nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
    def forward(self,x): return self.net(x)
class TinyUNet(nn.Module):
    def __init__(self,n=5,b=16):
        super().__init__(); self.e1=ConvBlock(3,b); self.e2=ConvBlock(b,b*2); self.bot=ConvBlock(b*2,b*4); self.d2=ConvBlock(b*4+b*2,b*2); self.d1=ConvBlock(b*2+b,b); self.out=nn.Conv2d(b,n,1)
    def forward(self,x):
        a=self.e1(x); b=self.e2(F.max_pool2d(a,2)); c=self.bot(F.max_pool2d(b,2)); u=F.interpolate(c,size=b.shape[-2:],mode='bilinear',align_corners=False); u=self.d2(torch.cat([u,b],1)); u=F.interpolate(u,size=a.shape[-2:],mode='bilinear',align_corners=False); return self.out(self.d1(torch.cat([u,a],1)))
def dice_loss(logits,y,nc=5,eps=1e-6):
    p=logits.softmax(1); oh=F.one_hot(y,nc).permute(0,3,1,2).float(); inter=(p*oh).sum((0,2,3)); den=(p+oh).sum((0,2,3)); return 1-((2*inter+eps)/(den+eps)).mean()
def evaluate_seg(model,loader):
    model.eval(); cm=np.zeros((5,5),np.int64); losses=[]
    with torch.no_grad():
        for x,y,_ in loader:
            x,y=x.to(DEVICE),y.to(DEVICE); z=model(x); losses.append(float((F.cross_entropy(z,y)+dice_loss(z,y)).item())); pr=z.argmax(1).cpu().numpy(); gt=y.cpu().numpy()
            for a,b in zip(gt.reshape(-1),pr.reshape(-1)): cm[a,b]+=1
    iou=np.diag(cm)/(cm.sum(1)+cm.sum(0)-np.diag(cm)+1e-9); dice=2*np.diag(cm)/(cm.sum(1)+cm.sum(0)+1e-9); return {'loss':np.mean(losses),'iou':iou,'dice':dice,'miou':np.nanmean(iou),'pixel_acc':np.diag(cm).sum()/cm.sum(),'cm':cm}
train_dl=DataLoader(PetroDataset(splits['train'],CFG.IMAGE_SIZE),batch_size=CFG.BATCH_SIZE,shuffle=True,num_workers=0); val_dl=DataLoader(PetroDataset(splits['val'],CFG.IMAGE_SIZE),batch_size=CFG.BATCH_SIZE); test_dl=DataLoader(PetroDataset(splits['test'],CFG.IMAGE_SIZE),batch_size=CFG.BATCH_SIZE)
model=TinyUNet().to(DEVICE); opt=torch.optim.Adam(model.parameters(),lr=1e-3); history=[]; best=1e9; ckpt=Path(CFG.OUTPUT_DIR)/'tiny_unet.pt'
for ep in range(CFG.NUM_EPOCHS):
    model.train(); ls=[]
    for x,y,_ in train_dl:
        x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(); z=model(x); loss=F.cross_entropy(z,y)+dice_loss(z,y); loss.backward(); opt.step(); ls.append(loss.item())
    vm=evaluate_seg(model,val_dl); history.append({'epoch':ep+1,'train_loss':np.mean(ls),'val_loss':vm['loss'],'val_miou':vm['miou']}); print(history[-1])
    if vm['loss']<best: best=vm['loss']; torch.save(model.state_dict(),ckpt)
model.load_state_dict(torch.load(ckpt,map_location=DEVICE)); test_metrics=evaluate_seg(model,test_dl); print('TEST mIoU',round(test_metrics['miou'],3),'pixel acc',round(test_metrics['pixel_acc'],3)); print('Per-class IoU',dict(zip(SEMANTIC_NAMES.values(),np.round(test_metrics['iou'],3))))
pd.DataFrame(history).to_csv(Path(CFG.OUTPUT_DIR)/'segmentation_history.csv',index=False)

### Segmentation visual diagnostics

In [ ]:
x,y,names=next(iter(test_dl)); model.eval();
with torch.no_grad(): pr=model(x.to(DEVICE)).argmax(1).cpu().numpy()
fig,ax=plt.subplots(1,4,figsize=(15,4)); rgb=(x[0].numpy().transpose(1,2,0)*255).astype(np.uint8); gt=y[0].numpy(); pp=pr[0]
ax[0].imshow(rgb); ax[0].set_title('RGB'); ax[1].imshow(PALETTE[gt]); ax[1].set_title('Ground truth'); ax[2].imshow(PALETTE[pp]); ax[2].set_title('U-Net prediction'); ax[3].imshow(gt!=pp,cmap='magma'); ax[3].set_title('Error map')
for a in ax:a.axis('off'); plt.tight_layout(); plt.savefig(Path(CFG.OUTPUT_DIR)/'segmentation_prediction.png',dpi=160); plt.show()

## 13. Instance segmentation via connected components + watershed
Instance segmentation answers: **“Which individual object does each pixel belong to?”** The baseline separates touching pore/grain regions using distance transform and marker-controlled watershed. Advanced Mask R-CNN/SAM can replace this stage without changing downstream tables.

In [ ]:
def semantic_to_instances(sem_mask,target_classes=(1,2,3,4),min_area=8):
    out=np.zeros_like(sem_mask,dtype=np.int32); oid=1
    for c in target_classes:
        binary=sem_mask==c
        if not binary.any(): continue
        dist=ndi.distance_transform_edt(binary); coords=feature.peak_local_max(dist,min_distance=4,labels=binary)
        markers=np.zeros_like(sem_mask,dtype=np.int32)
        for j,(yy,xx) in enumerate(coords,1): markers[yy,xx]=j
        if markers.max()==0: markers,_=ndi.label(binary)
        lab=segmentation.watershed(-dist,markers,mask=binary)
        for region in measure.regionprops(lab):
            if region.area<min_area: continue
            out[lab==region.label]=oid; oid+=1
    return out
pred_inst=semantic_to_instances(pp); fig,ax=plt.subplots(1,2,figsize=(9,4)); ax[0].imshow(PALETTE[pp]); ax[0].set_title('Semantic prediction'); ax[1].imshow(pred_inst,cmap='nipy_spectral'); ax[1].set_title(f'Instances: {pred_inst.max()}'); [a.axis('off') for a in ax]; plt.show()

## 14. Object-level feature extraction
Each instance becomes one row. Geometry, color, texture and local-neighborhood features make the classical classifier interpretable and provide quantitative petrography.

In [ ]:
from skimage.feature import graycomatrix, graycoprops

def object_features(rgb,inst,sem=None,image_id='image',source_file='',page=1):
    rows=[]; gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    props=measure.regionprops(inst,intensity_image=gray)
    centroids=np.array([p.centroid[::-1] for p in props]) if props else np.empty((0,2))
    for k,p in enumerate(props):
        y1,x1,y2,x2=p.bbox; mask=inst==p.label; pix=rgb[mask]; per=max(p.perimeter,1e-6); circ=4*np.pi*p.area/(per*per); aspect=p.major_axis_length/max(p.minor_axis_length,1e-6)
        crop=gray[y1:y2,x1:x2]; glcm=graycomatrix(cv2.resize(crop,(32,32)),[1],[0],256,symmetric=True,normed=True) if crop.size else None
        d=np.linalg.norm(centroids-centroids[k],axis=1) if len(centroids)>1 else np.array([999.]); d[k]=999 if len(d)>k else 999
        sem_class=int(np.bincount(sem[mask]).argmax()) if sem is not None and mask.any() else -1
        rows.append({'source_file':source_file,'page':page,'image_id':image_id,'object_id':int(p.label),'semantic_class':sem_class,'area':p.area,'perimeter':p.perimeter,'equiv_diameter':p.equivalent_diameter_area,'major_axis':p.major_axis_length,'minor_axis':p.minor_axis_length,'aspect_ratio':aspect,'eccentricity':p.eccentricity,'solidity':p.solidity,'circularity':circ,'orientation_rad':p.orientation,'centroid_x':p.centroid[1],'centroid_y':p.centroid[0],'bbox_x1':x1,'bbox_y1':y1,'bbox_x2':x2,'bbox_y2':y2,'mean_r':pix[:,0].mean(),'mean_g':pix[:,1].mean(),'mean_b':pix[:,2].mean(),'std_color':pix.std(),'glcm_contrast':graycoprops(glcm,'contrast')[0,0] if glcm is not None else 0,'glcm_homogeneity':graycoprops(glcm,'homogeneity')[0,0] if glcm is not None else 0,'nearest_object_distance':float(d.min())})
    return pd.DataFrame(rows)
feat_demo=object_features(rgb,pred_inst,pp,names[0]); display(feat_demo.head())

## 15. Object classification — classical ML, CNN, hybrid CNN–Transformer
To make classification labels reliable in the synthetic experiment, we extract features/patches from **ground-truth instances**, while source-group splits prevent leakage. At real inference time, predicted instances feed the same classifiers.

- **Model 1:** Random Forest on interpretable morphology/color/texture.
- **Model 2:** lightweight CNN on object patches.
- **Model 3:** simplified PoreViT-inspired hybrid: CNN local texture + Transformer context tokens + feature fusion. This is an educational hybrid, **not a reproduction of PoreViT**.

In [ ]:
gt_obj=pd.read_csv(Path(CFG.DATA_DIR)/'metadata'/'object_ground_truth.csv')
# Build object features from GT instances and assign GT class by object_id.
all_feat=[]
for _,row in image_meta.iterrows():
    p=Path(row.path); im=np.array(Image.open(p)); ins=np.load(Path(CFG.DATA_DIR)/'annotations'/f'{p.stem}_instance.npy'); semm=np.load(Path(CFG.DATA_DIR)/'annotations'/f'{p.stem}_semantic.npy')
    f=object_features(im,ins,semm,p.stem,str(p)); labels=gt_obj[gt_obj.image_id==p.stem][['object_id','class_id','class_name','source_group']]; all_feat.append(f.merge(labels,on='object_id',how='inner'))
obj_df=pd.concat(all_feat,ignore_index=True); feature_cols=['area','perimeter','equiv_diameter','major_axis','minor_axis','aspect_ratio','eccentricity','solidity','circularity','orientation_rad','mean_r','mean_g','mean_b','std_color','glcm_contrast','glcm_homogeneity','nearest_object_distance']
# source-level split inherited from image split
train_groups=set(splits['train'].source_group); val_groups=set(splits['val'].source_group); test_groups=set(splits['test'].source_group)
tr=obj_df.source_group.isin(train_groups); va=obj_df.source_group.isin(val_groups); te=obj_df.source_group.isin(test_groups)
rf=RandomForestClassifier(n_estimators=180,class_weight='balanced',random_state=CFG.RANDOM_SEED,n_jobs=-1).fit(obj_df.loc[tr,feature_cols],obj_df.loc[tr,'class_id'])
rf_pred=rf.predict(obj_df.loc[te,feature_cols]); rf_prob=rf.predict_proba(obj_df.loc[te,feature_cols]).max(1)
print('RF macro-F1:',round(f1_score(obj_df.loc[te,'class_id'],rf_pred,average='macro'),3),'balanced acc:',round(balanced_accuracy_score(obj_df.loc[te,'class_id'],rf_pred),3)); print(classification_report(obj_df.loc[te,'class_id'],rf_pred,zero_division=0))
imp=pd.Series(rf.feature_importances_,index=feature_cols).sort_values(ascending=False); imp.head(12).plot.barh(figsize=(7,5)); plt.gca().invert_yaxis(); plt.title('Random Forest feature importance'); plt.tight_layout(); plt.savefig(Path(CFG.OUTPUT_DIR)/'rf_feature_importance.png',dpi=150); plt.show()

In [ ]:
class ObjectPatchDataset(Dataset):
    def __init__(self,df,size=64): self.df=df.reset_index(drop=True); self.size=size
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; im=np.array(Image.open(r.source_file).convert('RGB')); x1,y1,x2,y2=[int(r[c]) for c in ['bbox_x1','bbox_y1','bbox_x2','bbox_y2']]; pad=5; x1=max(0,x1-pad);y1=max(0,y1-pad);x2=min(im.shape[1],x2+pad);y2=min(im.shape[0],y2+pad)
        crop=cv2.resize(im[y1:y2,x1:x2],(self.size,self.size)); x=torch.from_numpy(crop.transpose(2,0,1)).float()/255.; return x,torch.tensor(int(r.class_id)-1),r.image_id,int(r.object_id)
class TinyCNN(nn.Module):
    def __init__(self,n=7): super().__init__(); self.feat=nn.Sequential(nn.Conv2d(3,24,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.fc=nn.Linear(64,n)
    def forward(self,x,return_features=False): f=self.feat(x).flatten(1); return (self.fc(f),f) if return_features else self.fc(f)
class HybridCNNTransformer(nn.Module):
    def __init__(self,n=7,d=64):
        super().__init__(); self.cnn=TinyCNN(n); self.proj=nn.Linear(64,d); enc=nn.TransformerEncoderLayer(d_model=d,nhead=4,dim_feedforward=128,batch_first=True,dropout=.1); self.tr=nn.TransformerEncoder(enc,1); self.head=nn.Sequential(nn.Linear(d*2,96),nn.ReLU(),nn.Dropout(.15),nn.Linear(96,n))
    def forward(self,x):
        _,f=self.cnn(x,True); local=self.proj(f); # create contextual tokens by combining local token with batch context token
        context=local.mean(0,keepdim=True).expand_as(local); tokens=torch.stack([local,context],1); z=self.tr(tokens); return self.head(torch.cat([z[:,0],z[:,1]],1))
def train_classifier(net,train_df,val_df,epochs=2):
    net=net.to(DEVICE); opt=torch.optim.Adam(net.parameters(),1e-3); dl=DataLoader(ObjectPatchDataset(train_df),32,shuffle=True); vdl=DataLoader(ObjectPatchDataset(val_df),64)
    best=1e9; path=Path(CFG.OUTPUT_DIR)/(net.__class__.__name__+'.pt')
    for ep in range(epochs):
        net.train(); ls=[]
        for x,y,*_ in dl: x,y=x.to(DEVICE),y.to(DEVICE); opt.zero_grad(); z=net(x); l=F.cross_entropy(z,y); l.backward();opt.step();ls.append(l.item())
        net.eval();vl=[]
        with torch.no_grad():
            for x,y,*_ in vdl: vl.append(F.cross_entropy(net(x.to(DEVICE)),y.to(DEVICE)).item())
        v=np.mean(vl); print(net.__class__.__name__,ep+1,'train',np.mean(ls),'val',v)
        if v<best:best=v;torch.save(net.state_dict(),path)
    net.load_state_dict(torch.load(path,map_location=DEVICE)); return net
cnn=train_classifier(TinyCNN(),obj_df[tr],obj_df[va],2 if CFG.FAST_DEMO else 8)
hybrid=train_classifier(HybridCNNTransformer(),obj_df[tr],obj_df[va],2 if CFG.FAST_DEMO else 8)
def eval_cls(net,df):
    dl=DataLoader(ObjectPatchDataset(df),64); ys=[];ps=[];cs=[]
    net.eval()
    with torch.no_grad():
        for x,y,*_ in dl:
            prob=net(x.to(DEVICE)).softmax(1).cpu(); ys+=y.numpy().tolist(); ps+=prob.argmax(1).numpy().tolist(); cs+=prob.max(1).values.numpy().tolist()
    return np.array(ys)+1,np.array(ps)+1,np.array(cs)
for name,net in [('CNN',cnn),('Hybrid',hybrid)]:
    yy,ppp,cc=eval_cls(net,obj_df[te]); print(name,'macro-F1',round(f1_score(yy,ppp,average='macro'),3),'balanced acc',round(balanced_accuracy_score(yy,ppp),3))

## 16. Explainability
Classical ML uses permutation importance. CNN uses a simple gradient saliency map on the input patch (a lightweight Grad-CAM-equivalent diagnostic for the demo). The purpose is geological validation: confirm that decisions are driven by the object rather than irrelevant borders/background.

In [ ]:
perm=permutation_importance(rf,obj_df.loc[te,feature_cols],obj_df.loc[te,'class_id'],n_repeats=5,random_state=CFG.RANDOM_SEED,scoring='f1_macro')
pd.Series(perm.importances_mean,index=feature_cols).sort_values().tail(10).plot.barh(figsize=(7,5)); plt.title('Permutation importance'); plt.tight_layout(); plt.show()

def input_saliency(net,x):
    net.eval(); x=x.clone().detach().unsqueeze(0).to(DEVICE).requires_grad_(True); z=net(x); cls=z.argmax(1); z[0,cls].backward(); sal=x.grad.abs().max(1)[0][0].detach().cpu().numpy(); return sal,int(cls.item()+1)
ds=ObjectPatchDataset(obj_df[te]); xx,yy,*_=ds[0]; sal,pc=input_saliency(cnn,xx); fig,ax=plt.subplots(1,2,figsize=(7,3)); ax[0].imshow(xx.numpy().transpose(1,2,0));ax[0].set_title('Object patch');ax[1].imshow(sal,cmap='magma');ax[1].set_title(f'Saliency, pred={OBJECT_NAMES.get(pc)}');[a.axis('off') for a in ax];plt.show()

## 17. Uncertainty and human-in-the-loop queue
Low OCR confidence, low object-class confidence and model disagreement are exported for expert review. Corrections are append-only records suitable for later retraining.

In [ ]:
test_objects=obj_df[te].copy(); yy,cnn_pred,cnn_conf=eval_cls(cnn,test_objects); _,hy_pred,hy_conf=eval_cls(hybrid,test_objects); test_objects['rf_pred']=rf_pred;test_objects['rf_conf']=rf_prob;test_objects['cnn_pred']=cnn_pred;test_objects['cnn_conf']=cnn_conf;test_objects['hybrid_pred']=hy_pred;test_objects['hybrid_conf']=hy_conf
test_objects['model_disagreement']=(test_objects.rf_pred!=test_objects.cnn_pred)|(test_objects.cnn_pred!=test_objects.hybrid_pred); test_objects['uncertain']=(test_objects[['rf_conf','cnn_conf','hybrid_conf']].max(axis=1)<CFG.CONFIDENCE_THRESHOLD)|test_objects.model_disagreement
ocr_uncertain=pd.DataFrame(text_rows); ocr_uncertain=ocr_uncertain[(ocr_uncertain.method=='tesseract')&(ocr_uncertain.confidence<CFG.CONFIDENCE_THRESHOLD)].assign(case_type='ocr')
obj_unc=test_objects[test_objects.uncertain].copy(); obj_unc['case_type']='object_classification'; uncertainty=pd.concat([ocr_uncertain,obj_unc],ignore_index=True,sort=False); uncertainty.to_csv(Path(CFG.OUTPUT_DIR)/'uncertainty_queue.csv',index=False); display(uncertainty.head())

def save_human_correction(case_id,old_label,new_label,reviewer='geologist'):
    path=Path(CFG.OUTPUT_DIR)/'human_corrections.csv'; row=pd.DataFrame([{'case_id':case_id,'old_label':old_label,'new_label':new_label,'reviewer':reviewer,'timestamp':pd.Timestamp.utcnow().isoformat()}]); row.to_csv(path,mode='a',header=not path.exists(),index=False); return path
print('Example correction file path:',save_human_correction('demo_case','isolated_vug','connected_vug'))

## 18. Quantitative geological results
We calculate 2D visible porosity, object fractions, size distributions and fracture orientations. **2D visible porosity is not automatically laboratory porosity, effective porosity, or 3D connected porosity.**

In [ ]:
# Use GT/predicted semantic example for demonstration
visible_porosity=float((pp==2).mean()); frac_density=float((pp==3).sum()/pp.size); print('2D visible porosity:',round(visible_porosity,4),'fracture pixel fraction:',round(frac_density,4))
fig,ax=plt.subplots(1,3,figsize=(14,4));
obj_df[obj_df.class_id.isin([4,5,6])].equiv_diameter.hist(ax=ax[0],bins=20);ax[0].set_title('Pore/vug equivalent diameter');obj_df[obj_df.class_id.isin([1,2])].equiv_diameter.hist(ax=ax[1],bins=20);ax[1].set_title('Grain-size distribution');
angles=np.degrees(obj_df[obj_df.class_id==7].orientation_rad);ax[2].hist(angles,bins=18);ax[2].set_title('Fracture orientation');
plt.tight_layout();plt.savefig(Path(CFG.OUTPUT_DIR)/'quantitative_petrography.png',dpi=160);plt.show()

## 19. Integrated multimodal relational output
All outputs include provenance and model version. This is the central portfolio message: the system is not merely OCR or an image classifier; it is a traceable multimodal pipeline.

In [ ]:
MODEL_VERSION='geo-ai-demo-1.0'
predictions=test_objects[['source_file','image_id','object_id','class_name','rf_pred','rf_conf','cnn_pred','cnn_conf','hybrid_pred','hybrid_conf']].copy(); predictions['model_version']=MODEL_VERSION; predictions.to_csv(Path(CFG.OUTPUT_DIR)/'predictions.csv',index=False)
obj_df.to_csv(Path(CFG.OUTPUT_DIR)/'object_measurements.csv',index=False)
# Excel tables collected from native spreadsheet/docx structures
with pd.ExcelWriter(Path(CFG.OUTPUT_DIR)/'extracted_tables.xlsx') as writer:
    n=0
    for r in records:
        for t in r.get('tables',[]) or []:
            pd.DataFrame(t).to_excel(writer,sheet_name=f'table_{n:03d}'[:31],index=False); n+=1
print('Saved',n,'native tables')
# Minimal HTML report
html=(f"<html><body><h1>Geological AI Pipeline Summary</h1><p>Model version: {MODEL_VERSION}</p>" + f"<ul><li>Files discovered: {len(manifest)}</li><li>Unified records: {len(records)}</li>" + f"<li>Entities extracted: {len(entities)}</li><li>Petrographic objects: {len(obj_df)}</li>" + f"<li>Test semantic mIoU: {test_metrics['miou']:.3f}</li><li>2D visible porosity example: {visible_porosity:.3f}</li></ul>" + "<p><b>Caution:</b> synthetic-data performance is not field validation.</p></body></html>")
(Path(CFG.OUTPUT_DIR)/'final_summary_report.html').write_text(html)
print('Outputs:',sorted(p.name for p in Path(CFG.OUTPUT_DIR).iterdir()))

## 20. Model comparison
Metrics are populated only where a valid held-out source-level test set exists. OCR metrics remain `N/A` when Tesseract is unavailable; this avoids inventing accuracy.

In [ ]:
yy,cnn_p,cnn_c=eval_cls(cnn,obj_df[te]); _,hy_p,hy_c=eval_cls(hybrid,obj_df[te])
comparison=pd.DataFrame([
['Text layer / Tesseract','OCR','page image','text+boxes','N/A unless OCR+GT available','fast','traceable baseline','scan quality/language dependent','high'],
['TF-IDF + LogisticRegression','document classification','OCR text','page class','held-out source split','very fast','interpretable','text-only','high'],
['Tiny U-Net','semantic segmentation','RGB image','pixel classes',f"mIoU={test_metrics['miou']:.3f}",'fast','lightweight','synthetic domain gap','medium'],
['Watershed','instance separation','semantic mask','object IDs','not learned','very fast','no training','touching complex objects','medium'],
['Random Forest','object classification','morphology/color','object class',f"macroF1={f1_score(obj_df.loc[te,'class_id'],rf_pred,average='macro'):.3f}",'fast','interpretable','feature engineering','high'],
['Tiny CNN','object classification','object patch','object class',f"macroF1={f1_score(yy,cnn_p,average='macro'):.3f}",'fast','learns texture','less context','medium'],
['CNN + Transformer','object classification','patch+batch context','object class',f"macroF1={f1_score(yy,hy_p,average='macro'):.3f}",'moderate','local+context fusion','demo context simplified','research']
],columns=['method','task','input','output','accuracy_metrics','inference_time','advantages','limitations','deployment_suitability'])
comparison.to_csv(Path(CFG.OUTPUT_DIR)/'model_comparison.csv',index=False); display(comparison)

## 21. High-level single-function inference
The function below orchestrates discovery, ingestion, OCR/text extraction, entities, geological-image discovery, segmentation, instance separation, measurements, classification and exports. It reuses trained demo models when available. Folder mode works for arbitrary supported mixed files; unannotated real images receive inference only — no ground-truth accuracy is claimed.

In [ ]:
def run_geological_ai_pipeline(source='synthetic',input_path=None,output_path=None):
    input_path=input_path or CFG.DATA_DIR; output=Path(output_path or CFG.OUTPUT_DIR); output.mkdir(parents=True,exist_ok=True)
    mf=discover_files(input_path); recs=ingest_manifest(mf); texts=[]; ents=[]; obj_tables=[]; preds=[]
    for r in recs:
        o=extract_text_record(r); r['extracted_text']=o['text']; texts.append({'source_file':r['source_file'],'page':r['page'],'text':o['text'],'confidence':o['confidence'],'method':o['method']}); ents+=extract_entities(o['text'] or r.get('text_layer',''),r['source_file'],r['page'],o['confidence'] or .5)
        if r['kind']=='standalone_image':
            rgb0=r['image']; rs=cv2.resize(rgb0,(CFG.IMAGE_SIZE,CFG.IMAGE_SIZE)); x=torch.from_numpy(rs.transpose(2,0,1)).float().unsqueeze(0).to(DEVICE)/255.
            with torch.no_grad(): logits=model(x); sm=logits.argmax(1)[0].cpu().numpy(); prob=logits.softmax(1).max(1).values[0].cpu().numpy()
            ins=semantic_to_instances(sm); ft=object_features(rs,ins,sm,Path(r['source_file']).stem,r['source_file'],r['page'])
            if len(ft):
                ft['model_version']=MODEL_VERSION; obj_tables.append(ft)
                common=[c for c in feature_cols if c in ft.columns]
                if len(common)==len(feature_cols):
                    pc=rf.predict(ft[feature_cols]); cf=rf.predict_proba(ft[feature_cols]).max(1); q=ft[['source_file','page','image_id','object_id']].copy(); q['predicted_class_id']=pc;q['predicted_class']=[OBJECT_NAMES.get(int(v),'other') for v in pc];q['confidence']=cf;q['model_version']=MODEL_VERSION;preds.append(q)
            ann=PALETTE[sm]; Image.fromarray(ann).save(output/(Path(r['source_file']).stem+'_semantic.png'))
    entdf=pd.DataFrame(ents); objdf=pd.concat(obj_tables,ignore_index=True) if obj_tables else pd.DataFrame(); preddf=pd.concat(preds,ignore_index=True) if preds else pd.DataFrame()
    mf.to_csv(output/'document_manifest.csv',index=False); entdf.to_csv(output/'geological_entities.csv',index=False); objdf.to_csv(output/'object_measurements.csv',index=False); preddf.to_csv(output/'predictions.csv',index=False); (output/'extracted_text.json').write_text(json.dumps(texts,indent=2))
    return {'manifest':mf,'records':recs,'entities':entdf,'objects':objdf,'predictions':preddf,'metrics':{'semantic_test':test_metrics},'output_dir':str(output)}
# Example (uncomment to rerun orchestration):
# results = run_geological_ai_pipeline(source=CFG.DATA_SOURCE, input_path=CFG.DATA_DIR, output_path=CFG.OUTPUT_DIR)

## 22. Switch to real folder mode
After the synthetic run succeeds, change **only** these values and rerun ingestion/inference cells:

```python
CFG.DATA_SOURCE = "folder"
CFG.DATA_DIR = "/content/geological_data"
results = run_geological_ai_pipeline(
    source="folder",
    input_path=CFG.DATA_DIR,
    output_path=CFG.OUTPUT_DIR
)
```

For unannotated real petrographic images, the notebook runs inference and object-feature analysis only. It **does not** calculate accuracy because no ground truth exists.

## 23. Real-data adaptation guide
1. **Upload documents** into one folder; recursive discovery supports PDF/DOCX/XLSX/CSV/JPG/PNG/TIFF.
2. **Petrographic images:** place full-resolution images anywhere under the folder; keep original source IDs.
3. **Semantic annotations:** store one integer mask per source image, same spatial size, class IDs documented in a label map.
4. **Instance annotations:** store unique integer object IDs or COCO JSON polygons/RLE.
5. **New classes:** extend the label map only after a geologist defines reproducible visual criteria.
6. **Fine-tuning segmentation:** split by source image/well/core first, then create patches/augmentations. Never split patches from one source across train/test.
7. **Fine-tuning object classification:** derive patches only after the source split; maintain object-to-source provenance.
8. **Class imbalance:** use class weights/focal loss, balanced sampling, macro-F1 and per-class recall.
9. **Geologist validation:** review low-confidence/error clusters, not only random images; preserve corrections as versioned labels.
10. **Domain shift:** compare image histograms, resolution, modality, feature embeddings and error rates by scanner/lab/formation.
11. **2D → 3D micro-CT:** replace 2D U-Net with 3D U-Net/sliding-window inference; features become voxel volume, surface area, sphericity, connectivity, pore-throat/network metrics.
12. **Deployment:** version OCR/layout/segmentation/classification models independently and store model version on every prediction.

## 24. Interview-ready explanation
### 30 seconds
> “I built a traceable multimodal geological-AI pipeline that handles both legacy documents and petrographic imagery. It distinguishes searchable PDFs from scans, applies OCR and structured geological extraction, then analyzes geological images with semantic and instance segmentation. Each pore, grain or fracture receives an object ID, quantitative measurements, a class and confidence, while provenance links every result back to the original file and page. The default notebook is reproducible on synthetic data and is designed to be adapted to expert-labelled field data.”

### 2 minutes
> “The architecture has two connected branches. The document branch performs ingestion, scan QC, layout analysis, OCR, table/entity extraction and page classification. The imaging branch performs QC, U-Net semantic segmentation, watershed instance separation, morphology/texture extraction and object classification using interpretable ML, a CNN and a simplified CNN–Transformer hybrid. The important engineering decision is traceability: the final schema connects source document, page, extracted image, object ID, prediction, confidence and model version. I also enforce source-level train/validation/test splitting before patch generation, export an uncertainty queue for geologist review, and never claim accuracy for unlabelled real data. Synthetic data demonstrate the complete software path; real deployment would require domain adaptation and expert-labelled Saudi carbonate data.”

### 5-minute structure
1. Business problem: heterogeneous legacy knowledge + physical rock assets.
2. Document pipeline: searchable vs raster → preprocessing → layout → OCR → tables/entities → classification.
3. Image pipeline: semantic pixels → watershed/advanced instance masks → object features → class/confidence.
4. Validation: source-level split, per-class metrics, explainability, uncertainty/HITL.
5. Integration: provenance graph/relational schema.
6. Limitations: synthetic-to-real gap, OCR quality, 2D vs 3D petrophysical meaning.
7. Next step: expert-labelled real documents/images, stronger layout model, Mask R-CNN/SAM, 3D U-Net, multimodal retrieval.

### Main novelty
**The novelty is integration and traceability, not claiming a new foundation model:** heterogeneous documents and rock images are converted into linked, auditable, object-level geological records.

### Ten likely interview questions
1. **Why OCR before RAG?** RAG needs machine-readable evidence; OCR/layout/table reconstruction digitize raster documents first.
2. **Semantic vs instance segmentation?** Semantic assigns a class to pixels; instance segmentation separates individual Grain 1/2/3 objects.
3. **Why watershed?** It is a transparent baseline for splitting touching objects after semantic segmentation.
4. **Why source-level splitting?** Patch-level random splitting leaks near-duplicate information from one source image/document into test data.
5. **Why macro-F1?** Geological object classes are imbalanced; macro-F1 weights minority classes equally.
6. **Why hybrid CNN–Transformer?** CNN captures local texture; attention can incorporate broader object/neighborhood context.
7. **Why not claim real accuracy?** Synthetic performance measures software/model behavior on the synthetic distribution only.
8. **How do you validate geologically?** Compare against expert masks/classes, independent measurements, error clusters and uncertainty review.
9. **What is the biggest deployment risk?** Domain shift across scanners, staining/epoxy, formations, resolutions and document quality.
10. **How would you extend to micro-CT?** 3D U-Net/instance methods, voxel-level provenance and 3D connectivity/pore-network metrics.